# 폰 모델(Gemma 4 E2B) 깨우기 대화 미세조정

이 노트북은 **Google Colab**에서 위에서부터 한 칸씩 실행합니다 (각 칸 왼쪽 ▶ 버튼, 또는 Shift+Enter).

| 단계 | 하는 일 | 걸리는 시간(대략) |
|---|---|---|
| 0 | GPU 켜기 확인 | 1분 |
| 1 | 필요한 프로그램 설치 | 5분 |
| 2 | 프로젝트 코드와 학습 데이터 받기 | 1분 |
| 3 | 원본 모델 불러오기 | 5분 |
| 4 | 학습 **전** 대답 보기 | 2분 |
| 5 | 데이터를 학습용으로 바꾸기 | 1분 |
| 6 | 학습 (LoRA) | 20~40분 |
| 7 | 학습 **후** 대답 보기 | 2분 |
| 8 | 합쳐서 저장 → 폰용 파일(.litertlm)로 변환 | 20~30분 |
| 9 | 구글 드라이브에 저장 | 5분 |

**시작 전에**: 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU**(또는 더 좋은 GPU)를 고르세요.

용어
- **LoRA**: 모델 전체(약 50억 개 숫자)를 고치지 않고, 작은 '덧붙이는 판'만 학습하는 방법. 무료 GPU로도 됩니다.
- **학습 데이터**: `tools/finetune/make_wake_data.py`가 만든 가상 통화. 앱이 실제로 AI에 보내는 것과 같은 형식이고, 대화 엔진의 검사를 통과한 대답만 들어 있습니다.

## 0. GPU 확인
아래 칸을 실행해서 `Tesla T4` 같은 GPU 이름이 나오면 됩니다. 오류가 나면 런타임 유형을 GPU로 바꾸세요.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. 프로그램 설치
- `unsloth`: 적은 GPU 메모리로 빠르게 학습하게 해 주는 도구
- `litert-torch-nightly`: 학습한 모델을 폰용 파일(.litertlm)로 바꾸는 Google 공식 도구. 학습 도구와 버전이 부딪히지 않게 `uv tool`로 따로 설치합니다.

In [ ]:
!pip install -q unsloth
!pip install -q uv
!uv tool install -q litert-torch-nightly
!uv tool install -q litert-lm
import os
os.environ["PATH"] += ":/root/.local/bin"
!litert-torch --help | head -5

## 2. 프로젝트 코드와 학습 데이터 받기
`morning-gemma-finetune.zip`을 Colab 왼쪽 **파일(📁)** 창에 올려 두었으면 그걸 쓰고, 없으면 GitHub에서 받습니다.
대화 엔진(`engine/dialog.py`)은 학습 전후 대답을 비교할 때 앱과 같은 프롬프트를 만드는 데 씁니다.

In [ ]:
import os, sys, json, random
if os.path.exists("morning-gemma-finetune.zip"):
    !rm -rf bundle && unzip -q -o morning-gemma-finetune.zip -d bundle
    ROOT = "bundle"
else:
    !rm -rf morning-call-ai && git clone -q https://github.com/SM0625/morning-call-ai
    ROOT = "morning-call-ai"
sys.path.insert(0, f"{ROOT}/engine")
import dialog

DATA = f"{ROOT}/tools/finetune/wake_train.jsonl"
calls = [json.loads(line) for line in open(DATA, encoding="utf-8")]
random.Random(0).shuffle(calls)
n_val = max(10, len(calls) // 10)          # 10%는 학습에 쓰지 않고 확인용으로 남긴다
val_calls, train_calls = calls[:n_val], calls[n_val:]
answers = sum(m["role"] == "assistant" for c in train_calls for m in c["messages"])
print(f"통화 {len(calls)}개 → 학습 {len(train_calls)}개(대답 {answers}개), 확인용 {len(val_calls)}개")
print(json.dumps(train_calls[0]["messages"][1:4], ensure_ascii=False, indent=1))

## 3. 원본 모델 불러오기
- `load_in_4bit=True`: 모델을 4비트로 줄여 메모리를 아낍니다(T4 16GB에 맞추려고). A100 같은 큰 GPU면 `False`로 바꿔도 됩니다.
- 모델 이름 `unsloth/gemma-4-E2B-it`은 Google 원본(`google/gemma-4-E2B-it`)과 같은 가중치이고 로그인 없이 받을 수 있습니다.

In [ ]:
from unsloth import FastModel
import torch

MAX_LEN = 4096   # 시스템 프롬프트(약 1,000토큰) + 여러 턴 대화가 들어가야 한다
model, processor = FastModel.from_pretrained(
    "unsloth/gemma-4-E2B-it",
    max_seq_length=MAX_LEN,
    load_in_4bit=True,
    full_finetuning=False,
)
tok = getattr(processor, "tokenizer", processor)   # 글자만 다루는 토크나이저

## 4. 학습 **전** 대답 보기
확인용 통화(학습에 안 쓰는 것)에서 사용자 말 몇 개를 골라, 지금 모델이 어떻게 대답하는지 봅니다.
폰 앱처럼 **생각(thinking)은 끄고** 대답하게 합니다.

In [ ]:
def ask(messages, max_new_tokens=120):
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    ids = tok(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    out = model.generate(**ids, max_new_tokens=max_new_tokens, do_sample=True, temperature=0.7, top_p=0.95, top_k=40)
    return tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True).strip()

# 확인용 통화에서 '사용자 말 + 그 앞까지의 대화'를 몇 개 뽑는다
probes = []
for c in val_calls[:6]:
    m = c["messages"]
    i = random.Random(len(m)).choice([k for k in range(2, len(m)) if m[k]["role"] == "user"])
    probes.append(m[:i + 1])

FastModel.for_inference(model)
before = [ask(p) for p in probes]
for p, b in zip(probes, before):
    print("👤", p[-1]["content"].split("\n")[0], "\n🤖(전)", b, "\n")

## 5. 데이터를 학습용으로 바꾸기
대화 전체를 모델이 읽는 글자로 바꾸고, **AI가 한 말(assistant)만** 배우게 표시합니다.
(시스템 프롬프트와 사용자 말은 '읽기만' 하고 따라 하지 않게 합니다.)

In [ ]:
def encode(messages):
    """input_ids와 labels. labels는 AI 대답 부분만 남기고 나머지는 -100(배우지 않음)"""
    text = tok.apply_chat_template(messages, tokenize=False, enable_thinking=False)
    spans = []
    for i, m in enumerate(messages):
        if m["role"] != "assistant":
            continue
        before = tok.apply_chat_template(messages[:i], tokenize=False, add_generation_prompt=True, enable_thinking=False)
        upto = tok.apply_chat_template(messages[:i + 1], tokenize=False, enable_thinking=False)
        assert text.startswith(before) and text.startswith(upto), "대화 템플릿이 예상과 달라요"
        spans.append((len(before), len(upto)))
    enc = tok(text, add_special_tokens=False, return_offsets_mapping=True, truncation=True, max_length=MAX_LEN)
    labels = [tid if any(a <= s < b for a, b in spans) else -100
              for tid, (s, e) in zip(enc["input_ids"], enc["offset_mapping"])]
    return {"input_ids": enc["input_ids"], "attention_mask": enc["attention_mask"], "labels": labels}

from datasets import Dataset
train_ds = Dataset.from_list([encode(c["messages"]) for c in train_calls])
val_ds = Dataset.from_list([encode(c["messages"]) for c in val_calls])
lengths = [len(x) for x in train_ds["input_ids"]]
print(f"학습 {len(train_ds)}개, 길이 최대 {max(lengths)} 토큰, 평균 {sum(lengths)//len(lengths)} 토큰")
learned = sum(l != -100 for l in train_ds[0]["labels"])
print(f"첫 통화에서 배우는 토큰 {learned}개 / 전체 {len(train_ds[0]['labels'])}개")

## 6. 학습 (LoRA)
- `r=16`: 덧붙이는 판의 크기. 크면 더 많이 배우지만 데이터가 적으면 외워 버립니다.
- `learning_rate=1e-4`, 2바퀴(epoch): 너무 세게 배우면 원래 한국어 실력이 망가질 수 있어 약하게 시작합니다.
- 화면의 `loss`(틀린 정도)가 점점 줄어들면 잘 배우는 중입니다. `eval_loss`가 다시 올라가기 시작하면 외우기 시작한 것입니다.

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers=False, finetune_language_layers=True,   # 글자만 다루므로 그림 부분은 건드리지 않는다
    finetune_attention_modules=True, finetune_mlp_modules=True,
    r=16, lora_alpha=16, lora_dropout=0, bias="none", random_state=3407,
)

from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bf16_supported

FastModel.for_training(model)
trainer = Trainer(
    model=model,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=DataCollatorForSeq2Seq(tok, padding=True, label_pad_token_id=-100),
    args=TrainingArguments(
        output_dir="lora-out",
        per_device_train_batch_size=1, gradient_accumulation_steps=8,
        num_train_epochs=2, learning_rate=1e-4, warmup_steps=10, lr_scheduler_type="linear",
        logging_steps=5, eval_strategy="steps", eval_steps=25, save_strategy="no",
        bf16=is_bf16_supported(), fp16=not is_bf16_supported(),
        optim="adamw_8bit", gradient_checkpointing=True, report_to="none", seed=3407,
    ),
)
trainer.train()

## 7. 학습 **후** 대답 보기
4번과 같은 질문에 다시 대답하게 해서 나란히 봅니다. 대화 엔진의 검사에 걸리는지도 표시합니다.

In [ ]:
FastModel.for_inference(model)
after = [ask(p) for p in probes]
for p, b, a in zip(probes, before, after):
    user = p[-1]["content"].split("\n")[0]
    stage = 4 if "단계: 마무리" in p[-1]["content"] else 0
    flag = lambda r: dialog.reply_issues(r, user, stage, False) or "통과"
    print("👤", user)
    print("🤖(전)", b, "  ←", flag(b))
    print("🤖(후)", a, "  ←", flag(a), "\n")

## 8. 합쳐서 저장 → 폰용 파일로 변환
1. LoRA 판을 원본에 합쳐 보통 모델(16비트)로 저장합니다.
2. Google 변환 도구로 `.litertlm`을 만듭니다.
   - `--jinja_chat_template_override`: **꼭 필요합니다.** 원본의 대화 템플릿을 그대로 넣으면 폰에서 첫 메시지부터
     `map has no method named get` 오류가 납니다. 폰용(litert-community) 템플릿으로 바꿔 넣습니다.
3. PC용 실행기(`litert-lm run`)로 한 번 대답시켜 봅니다.

무료 Colab은 메모리(RAM)가 약 12GB라 변환 중에 멈출 수 있습니다. 그러면 **런타임 → 런타임 유형 변경 → 고용량 RAM**(Colab Pro)으로 바꾸거나,
합친 모델(`merged/`)을 드라이브에 저장해 두고 다른 컴퓨터에서 변환하세요.

In [ ]:
model.save_pretrained_merged("merged", processor, save_method="merged_16bit")
!ls -la merged | head -20

In [ ]:
!rm -rf out && litert-torch export_hf \
    --model=merged \
    --output_dir=out \
    --externalize_embedder \
    --jinja_chat_template_override=litert-community/gemma-4-E2B-it-litert-lm
!ls -la out

In [ ]:
!litert-lm run out/model.litertlm --prompt="5분만 더 잘래" 2>&1 | tail -5

## 9. 구글 드라이브에 저장
`내 드라이브/morning-call/morning-gemma.litertlm`으로 저장합니다. PC로 내려받은 뒤 폰에 넣는 방법은 README의 "미세조정한 모델 쓰기"를 보세요.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/morning-call
!cp out/model.litertlm /content/drive/MyDrive/morning-call/morning-gemma.litertlm
!ls -la /content/drive/MyDrive/morning-call